# Retrival Notebook 
  
Dieses Notebook dient dazu die Embeddings aus 04_embeddings.ipynb zu vergleichen.

In [ ]:

import faiss
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


import sys
from pathlib import Path

PROJECT_ROOT = next(d for d in (Path.cwd(), *Path.cwd().parents)
                    if (d / "config.yaml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import embedding_name, load_config, paths

CFG = load_config(PROJECT_ROOT)
PATHS = paths(CFG, PROJECT_ROOT)

METHOD = CFG["vpr"]["method"]
MODEL_ID = CFG["vpr"]["models"][METHOD]
RETRIEVAL_METHOD = CFG["retrieval"]["method"]
ADAPTER = CFG["vpr"].get("adapter", "none")
EMBEDDING_DIR = PATHS.embedding_dir(METHOD)
RETRIEVAL_DIR = PATHS.retrieval / METHOD
RETRIEVAL_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_NAME = embedding_name(CFG)

embedding_path = EMBEDDING_DIR / f"{EMBEDDING_NAME}_embeddings.npy"
metadata_path = EMBEDDING_DIR / f"{EMBEDDING_NAME}_metadata.parquet"

from src.run_guard import (
    embedding_fingerprint,
    print_run_header,
    require_fingerprint,
    validate_config,
    write_fingerprint,
)

validate_config(CFG)

embedding_metadata = pd.read_parquet(metadata_path)

# Bricht ab, statt eine Datei aus einem anderen Lauf zu verwenden.
FINGERPRINT = embedding_fingerprint(CFG, METHOD, ADAPTER, embedding_metadata)
require_fingerprint(embedding_path, FINGERPRINT, what="Embeddings")

# Nur als memmap: 70 % der Zeilen sind train und werden hier nie gebraucht.
# Vollstaendig geladen waeren es bei MegaLoc 11 GB plus Kopien -- das
# sprengte 16 GB. So kommen nur database und query in den Speicher.
embeddings = np.load(embedding_path, mmap_mode="r")

print_run_header(CFG, "06_retrieval")
print("Embeddings")
print(f"Shape:      {embeddings.shape}")
print(f"DataType:   {embeddings.dtype}")
print()
print("Metadata")
print(f"Shape:      {embedding_metadata.shape}")

# Trennen und Assertions

In [ ]:
database_mask = (embedding_metadata["split"] == "database").to_numpy()
query_mask = (embedding_metadata["split"] == "query").to_numpy()

database_embeddings = np.ascontiguousarray(embeddings[np.flatnonzero(database_mask)], dtype=np.float32)
query_embeddings = np.ascontiguousarray(embeddings[np.flatnonzero(query_mask)], dtype=np.float32)

database_metadata = embedding_metadata[database_mask].reset_index(drop=True)
query_metadata = embedding_metadata[query_mask].reset_index(drop=True)


assert len(embeddings) == len(embedding_metadata)
assert len(database_embeddings) == len(database_metadata)
assert len(query_embeddings) == len(query_metadata)


print(f"Database Shape:     {database_embeddings.shape}")
print(f"Query Shape:        {query_embeddings.shape}")

# Retrieval Funktionen

In [ ]:
TOP_K = CFG["retrieval"]["top_k"]
CHUNK = 1024


all_indices = np.empty((len(query_embeddings), TOP_K), dtype=np.int64)
all_similarities = np.empty((len(query_embeddings), TOP_K), dtype=np.float32)

if RETRIEVAL_METHOD == "faiss":
    # Exakte Suche ueber das innere Produkt. Die Embeddings sind
    # L2-normalisiert, damit entspricht das der Cosinus-Aehnlichkeit.
    index = faiss.IndexFlatIP(database_embeddings.shape[1])
    index.add(database_embeddings)
    for start in tqdm(range(0, len(query_embeddings), CHUNK), desc="Retrieval (faiss)"):
        chunk = query_embeddings[start : start + CHUNK]
        scores, idx = index.search(chunk, TOP_K)
        all_indices[start : start + len(chunk)] = idx
        all_similarities[start : start + len(chunk)] = scores

elif RETRIEVAL_METHOD == "numpy":
    for start in tqdm(range(0, len(query_embeddings), CHUNK), desc="Retrieval (numpy)"):
        chunk = query_embeddings[start : start + CHUNK]
        sims = chunk @ database_embeddings.T
        part = np.argpartition(-sims, TOP_K - 1, axis=1)[:, :TOP_K]
        part_scores = np.take_along_axis(sims, part, axis=1)
        order = np.argsort(-part_scores, axis=1)
        all_indices[start : start + len(chunk)] = np.take_along_axis(
            part, order, axis=1
        )
        all_similarities[start : start + len(chunk)] = np.take_along_axis(
            part_scores, order, axis=1
        )
else:
    raise ValueError(f"Unbekannte Retrieval-Methode: {RETRIEVAL_METHOD}")

print(f"indices:      {all_indices.shape}")
print(f"similarities: {all_similarities.shape}")


In [ ]:
np.savez(
    RETRIEVAL_DIR / f"{EMBEDDING_NAME}_retrieval.npz",
    indices=all_indices,
    similarities=all_similarities
)

write_fingerprint(
    RETRIEVAL_DIR / f"{EMBEDDING_NAME}_retrieval.npz",
    FINGERPRINT,
    retrieval_method=RETRIEVAL_METHOD,
    top_k=TOP_K,
)

print(f"Saved to {RETRIEVAL_DIR}")
